In [1]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import os

from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *

# ================= 配置与数据加载 =================
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
RANDOM_SEED = 1234
TRAIN_PERCENTAGE = 0.80
EVALUATION_CUTOFF = 20

Tensorflow is not available


C:\Users\IceCould\.conda\envs\RecSysFramework\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)


In [3]:


# ================= 全局配置 =================
N_ROUNDS = 5        # 进行多少轮“对决”
EVALUATION_CUTOFF = 20  # 评估指标的 cutoff，比如 Recall@20

# 候选参数池
rp3_params_list = [
    {"topK": 127, "alpha": 0.858301729, "beta": 0.328691467, "normalize_similarity": True},
    {"topK": 138, "alpha": 1.239267955, "beta": 0.450766737, "normalize_similarity": True},
    {"topK": 297, "alpha": 1.893380004, "beta": 0.319763073, "normalize_similarity": True},
    {"topK": 251, "alpha": 1.218008572, "beta": 0.12950515, "normalize_similarity": True},
    {"topK": 418, "alpha": 1.892805041, "beta": 0.159176072, "normalize_similarity": True},
]


def run_round_based_tuning_recall(recommender_class, params_list, urm_all, model_name="Model", n_rounds=5):
    """
    执行多轮对决参数选择策略 (基于 Recall@20)
    """
    print(f"\n==================================================")
    print(f"开始 {model_name} 的多轮对决 (Rounds: {n_rounds})")
    print(f"指标: Recall@{EVALUATION_CUTOFF}")
    print(f"==================================================")

    # 初始化记分板：winning_counts[i] 表示第 i 组参数赢了多少轮
    winning_counts = [0] * len(params_list)

    for round_idx in range(n_rounds):
        print(f"\n>>> ROUND : {round_idx + 1} / {n_rounds}")

        # 1. 每一轮都重新随机划分数据
        # 第一次划分：分离出用于本轮“决胜”的 Test 集 (20%)
        URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(urm_all, train_percentage=0.8)

        # 第二次划分：分离出用于训练的 Train (80% of 80% = 64%)
        # 实际上我们用 URM_train 来 fit 模型，在 URM_test 上验证
        URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage=0.8)

        # 初始化本轮的评估器 (针对本轮的随机 Test 集)
        evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[EVALUATION_CUTOFF])

        round_winner_index = None
        max_recall = -1.0

        # 2. 遍历所有候选参数
        for i, params in enumerate(params_list):
            try:
                # 初始化模型
                recommender = recommender_class(URM_train)

                # 训练模型
                recommender.fit(**params)

                # 3. 在 Test 集上评估
                results_df, _ = evaluator_test.evaluateRecommender(recommender)

                # === 关键修改：抓取 RECALL 而不是 MAP ===
                # results_df 的索引是 cutoff 值，列是指标名称
                current_recall = results_df.loc[EVALUATION_CUTOFF]["RECALL"]

                print(f"   Config {i}: Recall@{EVALUATION_CUTOFF} = {current_recall:.6f}")

                # 记录本轮最高 Recall
                if current_recall > max_recall:
                    max_recall = current_recall
                    round_winner_index = i

            except Exception as e:
                print(f"   Config {i} Failed: {str(e)}")
                continue

        # 4. 记录本轮胜者
        if round_winner_index is not None:
            winning_counts[round_winner_index] += 1
            print(f"   [Round {round_idx + 1} Winner] Config {round_winner_index} (Recall: {max_recall:.6f})")

    # ================= 决出最终赢家 =================
    print(f"\n--------------------------------------------------")
    print(f"{model_name} 对决结果 (基于 Recall@{EVALUATION_CUTOFF}):")
    for i, count in enumerate(winning_counts):
        print(f"Config {i}: 获胜 {count} 轮")

    # 找到获胜次数最多的索引
    final_winner_index = winning_counts.index(max(winning_counts))
    print(f"\n>>> 最终冠军配置 (Index {final_winner_index}):")
    print(f"{params_list[final_winner_index]}")
    print(f"==================================================\n")

    return params_list[final_winner_index]



# 1. 运行 RP3beta 的多轮对决
best_ials_params = run_round_based_tuning_recall(
    RP3betaRecommender,
    rp3_params_list,
    urm_all,
    model_name="RP3beta",
    n_rounds=N_ROUNDS
)


print("全量训练完成。")


开始 RP3beta 的多轮对决 (Rounds: 5)
指标: Recall@20

>>> ROUND : 1 / 5
EvaluatorHoldout: Ignoring 29 ( 0.1%) Users that have less than 1 test interactions
RP3betaRecommender: Similarity column 6969 (100.0%), 2416.88 column/sec. Elapsed time 2.88 sec
EvaluatorHoldout: Processed 27066 (100.0%) in 9.71 sec. Users per second: 2787
   Config 0: Recall@20 = 0.213021
RP3betaRecommender: Similarity column 6969 (100.0%), 2265.45 column/sec. Elapsed time 3.08 sec
EvaluatorHoldout: Processed 27066 (100.0%) in 10.07 sec. Users per second: 2688
   Config 1: Recall@20 = 0.211086
RP3betaRecommender: Similarity column 6969 (100.0%), 1775.18 column/sec. Elapsed time 3.93 sec
EvaluatorHoldout: Processed 27066 (100.0%) in 11.08 sec. Users per second: 2443
   Config 2: Recall@20 = 0.209049
RP3betaRecommender: Similarity column 6969 (100.0%), 1865.15 column/sec. Elapsed time 3.74 sec
EvaluatorHoldout: Processed 27066 (100.0%) in 10.64 sec. Users per second: 2545
   Config 3: Recall@20 = 0.210433
RP3betaRecommender